In [23]:
# ============================= INICIALIZAÇÃO ============================= #
from segment2d import *
from segment2d.config import cfg
import numpy as np
import csv
from matplotlib import pyplot as plt
from ipywidgets import interact 
import monai
import os
import torch
import nibabel as nib
from tqdm import tqdm
%matplotlib inline

In [16]:
# ============================= CONFIGURAÇÕES ============================= #

# input: Mudar o Checkpoint 
ckpt_path = r"./Monai_Auto3Dseg/monai_combine_01234_ActiveFocal_2103/dice_0.6753.ckpt" 

# input_info = """Input the path of the checkpoint to be evaluated: """
# ckpt_path = input(input_info)

def val_path(path, data_type) -> ValueError:
    if os.path.exists(path):
        print(f"{data_type} found at: {path}")
    else:
        raise ValueError(f"{data_type} not found at: {path}")
    
def print_table(data):
        
    # Find the max width for the first column
    max_key_length = max(len(row[0]) for row in data)

    # Print the table
    print(f"{'Parameter':<{max_key_length}} | Value")
    print("-" * (max_key_length + 50))  # Adjust width dynamically
    for row in data:
        print(f"{row[0]:<{max_key_length}} | {row[1]}")

test_data_path = os.path.join(*ckpt_path.split("\\")[:-1], "test.csv")
predict_dir = os.path.join(*ckpt_path.split("\\")[:-1], "prediction")
val_path(test_data_path, "Checkpoint")
val_path(test_data_path, "Checkpoint")
os.makedirs(predict_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class GetConfig:
    def __init__(self, ckpt_path):
        self.CKPT_PATH = ckpt_path
        self.NUM_CLASSES = 5
        self.TASK = ("01234", "train_full")
        self.WEIGHTS = [0.1, 2, 2, 17, 140]
        self.MODEL = FCDenseNet(in_channels=1, n_classes=self.NUM_CLASSES) # Data Indim Model?

        self._get_model()
        self._get_task_weights()

    def _get_model(self):

        # adicionar a função do dinamic import do napari
        # esc olher o modelo a partir do nome do ckpt

        model = torch.nn.Sequential(
            monai.networks.nets.UNet(
                spatial_dims=2,
                in_channels=1,
                out_channels=self.NUM_CLASSES,
                channels=(16, 32, 64, 128, 256),
                strides=(2, 2, 2, 2),
                num_res_units=2,
            ),
            torch.nn.Softmax(dim=1),
        )       
        
        self.MODEL = model
    
    def _get_task_weights(self):
        
        if self.CKPT_PATH.split("/")[-2].split("_")[-3] == "01234":
            self.TASK = ("01234", "train_full")
            self.WEIGHTS = [0.1, 2, 2, 17, 140]
            self.NUM_CLASSES = 5

        elif self.CKPT_PATH.split("/")[-2].split("_")[-3] == "0123":
            self.TASK = ("0123", "train_combine")
            self.WEIGHTS = [0.1, 2, 2, 17]
            self.NUM_CLASSES = 4

        elif self.CKPT_PATH.split("/")[-2].split("_")[-3] == "012":
            self.TASK = ("012", "train_combine")
            self.WEIGHTS = [0.1, 2, 2]
            self.NUM_CLASSES = 3

        else:
            raise ValueError("Task not found")

cfig = GetConfig(ckpt_path)

# ============================= PRINT CONFIGURAÇÕES ============================= #
data = [
    ["Checkpoint", cfig.CKPT_PATH],
    ["Weights", cfig.WEIGHTS],
    ["Num Classes", cfig.NUM_CLASSES],
    ["Task", cfig.TASK],
    ["Device", device]
] #Model: {cfg.MODEL} \n

print_table(data)

Checkpoint found at: test.csv
Checkpoint found at: test.csv
Parameter   | Value
-------------------------------------------------------------
Checkpoint  | ./Monai_Auto3Dseg/monai_combine_01234_ActiveFocal_2103/dice_0.6753.ckpt
Weights     | [0.1, 2, 2, 17, 140]
Num Classes | 5
Task        | ('01234', 'train_full')
Device      | cuda


In [17]:
# ============================= FUNÇÕES ============================= #
def preprocess_data(image_path) -> dict:
    data = {}
    image = nib.load(image_path)
    data["header"] = image.header
    image = image.get_fdata()
    image = min_max_normalize(image)

    padded_image, crop_index, padded_index = pad_background(image, dim2pad=cfg.DATA.DIM2PAD)
    # padded_mask = pad_background_with_index(mask, crop_index, padded_index, dim2pad=cfg.DATA.DIM2PAD)
    data["crop_index"] = crop_index
    data["padded_index"] = padded_index
    data["original_shape"] = image.shape
    batch_images = []
    for i in range(padded_image.shape[-1]):
        slice_inputs = padded_image[..., i : i + 1]  # shape (224, 224, 1)
        slices_image = torch.from_numpy(slice_inputs.transpose(-1, 0, 1))  # shape (1, 224, 224)
        batch_images.append(slices_image)

    batch_images = torch.stack(batch_images).float()  # shape (9,1, 224, 224)
    data["image"] = batch_images
    return data

def predict_data(data, segmenter, patient="P", mvo=True, task="01234") -> np.ndarray:
    probability_output = segmenter.predict_patches(data["image"])  # shape (n, 5, 128, 128)
    seg = np.argmax(probability_output, axis=1).transpose(1, 2, 0)  # shape (128, 128, n)
    seg = remove_small_elements(seg, min_size_remove=800)

    myo = np.sum(seg == 2) + np.sum(seg == 3) + np.sum(seg == 4)
    infarction = np.sum(seg == 3) + np.sum(seg == 4)
    frequency_infarction = infarction / myo

    if patient == "N" and frequency_infarction < 0.015:
        seg[seg == 3] = 2
        seg[seg == 4] = 2
    elif patient == "P" and not mvo:
        seg[seg == 4] = 3

    if task == "012":
        seg[seg == 4] = 2
        seg[seg == 3] = 2
        print("Task 012")
    elif task == "0123":
        seg[seg == 4] = 3
        print("Task 0123")
    elif task == "01234":
        print("Task 01234")
    else:
        raise ValueError("Task not found")

    invert_seg = invert_padding(data["original_shape"], seg, data["crop_index"], data["padded_index"])
    return invert_seg

def make_volume(ndarray, voxel_spacing):
    volume = np.prod(voxel_spacing) * (ndarray.sum())
    return volume

In [18]:
# ============================= TESTE ============================= #

# load test data
with open(test_data_path, mode="r") as f:
    reader = csv.DictReader(f)
    list_test_subject = [row["path"] for row in reader]

test_dataset = EMIDEC_Test_Loader(list_test_subject)

# load model
segmenter = Segmenter(
    cfig.MODEL,
    cfig.WEIGHTS,
    5,
    0.001,
    0.5,
    50,
)

# set model to evaluation mode
segmenter.eval()

# load checkpoint
segmenter = Segmenter.load_from_checkpoint(
    checkpoint_path=ckpt_path,
    model=cfig.MODEL,
    class_weight=cfig.WEIGHTS,
    num_classes=cfig.NUM_CLASSES,
    learning_rate=0.001,
    factor_lr=0.5,
    patience_lr=50,
)

# move model to device
segmenter = segmenter.to(device)

wrong_disease = ['Case_P087', 'Case_P010', 'Case_P017', 'Case_P100', 'Case_P051', 'Case_P026', 'Case_P031']
wrong_MVO = ["Case_P021", "Case_P015"]

In [19]:
# ============================= AVALIAÇÂO ============================= #
dice_scores = {"dice_myocardium": [], "dice_lv": [], "dice_mi": [], "dice_mvo": []}
dice_scores_no_radio = {"dice_myocardium": [], "dice_lv": [], "dice_mi": [], "dice_mvo": []}

for _, subj in tqdm(enumerate(list_test_subject)):
    
    id_patient = subj.split("/")[-3]
    test_image = nib.load(subj).get_fdata()
    mask_image = nib.load(subj.replace("Images", "Contours")).get_fdata()
    affine = nib.load(subj).affine
    header = nib.load(subj).header
    
    if "N" in id_patient:
        patient = "N"
    else:
        patient = "P"

    table = [
        ["PATIENT ID", id_patient],
        ["PATIENT TYPE", patient],
        ["TEST IMAGE", subj],
        ["MASK IMAGE", subj.replace("Images", "Contours")],
       # ["AFFINE", affine],
       # ["HEADER", header]
    ]
    
    # print_table(table)


    if id_patient in wrong_disease:
        patient = "N"
    is_MVO = False if np.sum(mask_image == 4) == 0 else True
    if id_patient in wrong_MVO:
        is_MVO = False

    data = preprocess_data(subj)

    seg = predict_data(
        data=data,
        segmenter=segmenter, 
        patient=patient, 
        mvo=is_MVO, 
        task=cfig.TASK[0]
        ).astype(np.uint8)

    seg_no_radio = predict_data(data, segmenter, patient="P", mvo=True, task=cfig.TASK[0]).astype(np.uint8)
    # save segmentation result to nii file in prediction folder
    # create nii file from data["header"] and affine
    seg_nii = nib.Nifti1Image(seg, affine=affine, header=header)
    seg_no_radio_nii = nib.Nifti1Image(seg_no_radio, affine=affine, header=header)
    nib.save(seg_nii, f"./prediction/{id_patient}.nii.gz")
    nib.save(seg_no_radio_nii, f"./prediction/{id_patient}_no_radio.nii.gz")
    # if id_patient != "Case_P094":
    #     continue

    # print("number of MI: ", np.sum(seg == 4)+np.sum(seg==3))

    dice_myo = dice_volume(mask_image, seg, class_index=2)
    dice_lv = dice_volume(mask_image, seg, class_index=1)
    dice_mi = dice_volume(mask_image, seg, class_index=3)
    dice_mvo = dice_volume(mask_image, seg, class_index=4)
    dice_scores["dice_myocardium"].append(dice_myo)
    dice_scores["dice_lv"].append(dice_lv)
    dice_scores["dice_mi"].append(dice_mi)
    dice_scores["dice_mvo"].append(dice_mvo)

    dice_myo_no_radio = dice_volume(mask_image, seg_no_radio, class_index=2)
    dice_lv_no_radio = dice_volume(mask_image, seg_no_radio, class_index=1)
    dice_mi_no_radio = dice_volume(mask_image, seg_no_radio, class_index=3)
    dice_mvo_no_radio = dice_volume(mask_image, seg_no_radio, class_index=4)
    dice_scores_no_radio["dice_myocardium"].append(dice_myo_no_radio)
    dice_scores_no_radio["dice_lv"].append(dice_lv_no_radio)
    dice_scores_no_radio["dice_mi"].append(dice_mi_no_radio)
    dice_scores_no_radio["dice_mvo"].append(dice_mvo_no_radio)

1it [00:00,  4.25it/s]

Task 01234
Task 01234
Task 01234
Task 01234


3it [00:00,  6.33it/s]

Task 01234
Task 01234
Task 01234
Task 01234


5it [00:00,  6.87it/s]

...

Task 01234
Task 01234
Task 01234
Task 01234
Task 01234
Task 01234
Task 01234
Task 01234
Task 01234
Task 01234


17it [00:01, 15.26it/s]

Task 01234
Task 01234
Task 01234
Task 01234
Task 01234
Task 01234
Task 01234
Task 01234
Task 01234
Task 01234


20it [00:01, 11.14it/s]

Task 01234
Task 01234
Task 01234
Task 01234


In [ ]:
# ============================= RESULTADOS ============================= #
# print the mean and std of dice scores
table = [
    ["Metric", "Mean Dice", "Std Dice"]
]

# Populate the table with values
for key in dice_scores.keys():
    mean_val = f"{np.mean(dice_scores[key]):0.4f}"
    std_val = f"{np.std(dice_scores[key]):0.4f}"
    table.append([key, mean_val, std_val])

# Find the max width for each column
col_widths = [max(len(str(row[i])) for row in table) for i in range(len(table[0]))]

# Print the table
print(" | ".join(f"{table[0][i]:<{col_widths[i]}}" for i in range(len(table[0]))))
print("-" * (sum(col_widths) + 6))

for row in table[1:]:
    print(" | ".join(f"{row[i]:<{col_widths[i]}}" for i in range(len(row))))

Metric          | Mean Dice | Std Dice
--------------------------------------
dice_myocardium | 0.8173    | 0.0615  
dice_lv         | 0.9241    | 0.0245  
dice_mi         | 0.6609    | 0.2625  
dice_mvo        | 0.6442    | 0.4411  


In [21]:
from scipy.stats import ttest_rel
import numpy as np

# Assuming you have two arrays of Dice scores for Model A and Model B
dice_scores_a = dice_scores["dice_mi"]
dice_scores_b = dice_scores_no_radio["dice_mi"]

# Perform a paired t-test
t_stat, p_value = ttest_rel(dice_scores_a, dice_scores_b)

print(f"t-statistic: {t_stat}")
print(f"p-value: {p_value}")

t-statistic: 2.516611478398108
p-value: 0.020991504671294518


In [ ]:
id_patient = "Case_P078"
test_image = nib.load(f"./emidec-dataset-1.0.1/{id_patient}/Images/{id_patient}.nii.gz").get_fdata()
mask_image = nib.load(f"./emidec-dataset-1.0.1/{id_patient}/Contours/{id_patient}.nii.gz").get_fdata()
seg = nib.load(f"./prediction/{id_patient}.nii.gz").get_fdata()

padded_image, crop_index, padded_index = pad_background(test_image, dim2pad=cfg.DATA.DIM2PAD)
mask_padded = pad_background_with_index(mask_image, crop_index, padded_index, dim2pad=cfg.DATA.DIM2PAD)
seg_padded = pad_background_with_index(seg, crop_index, padded_index, dim2pad=cfg.DATA.DIM2PAD)


def plot_image_mask_z(image, mask, z, prediction=None):
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(image[..., z], cmap="gray")
    ax[0].imshow(image[..., z], cmap="gray")
    ax[0].set_title("Prediction")
    # set color for jet cmap, pixel value 1 is red, 2 is green, 3 is blue, 4 is yellow
    ax[0].imshow(prediction[..., z], cmap="jet", alpha=0.3, vmin=0, vmax=5)

    if prediction is not None:
        ax[1].imshow(image[..., z], cmap="gray")
        ax[1].set_title("Ground Truth")
        ax[1].imshow(mask[..., z], cmap="jet", alpha=0.3, vmin=0, vmax=5)
    # off axis
    ax[0].axis("off")
    ax[1].axis("off")
    plt.show()

interact(lambda z: plot_image_mask_z(padded_image, mask_padded, z, seg_padded), z=(0, test_image.shape[-1] - 1))

interactive(children=(IntSlider(value=3, description='z', max=6), Output()), _dom_classes=('widget-interact',)…

<function __main__.<lambda>(z)>

In [29]:
ckpt_path

'./Monai_Auto3Dseg/monai_combine_01234_ActiveFocal_2103/dice_0.6753.ckpt'